# Scenario Schema Consistency Test

Run Scenario Schema **3 times on the same origin date** and compare:
- Scenario design and structure
- Price point forecasts
- Full price distributions (quantiles)
- Rationales

This reveals whether the agent's estimates are stable or show significant variance across runs.

## Setup

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from aieng.forecasting.models import LITE_MODEL
from aieng.forecasting.evaluation.prediction import ContinuousForecast
from energy_oil_forecasting.data import WTI_SERIES_ID, build_wti_multivariate_service
from energy_oil_forecasting.scenario_schema_enhanced import (
    build_wti_news_scenario_schema_enhanced_config,
    build_wti_scenario_schema_enhanced_predictor,
)

# Configuration
ORIGIN_DATE = pd.Timestamp.now().normalize()  # Runs for today
NUM_RUNS = 3
HORIZONS = [5, 10, 21]

# Setup
data_service = build_wti_multivariate_service()
print(f"Testing Scenario Schema on origin: {ORIGIN_DATE.date()}")
print(f"Horizons: {HORIZONS} business days")
print(f"Number of runs: {NUM_RUNS}")

Testing Scenario Schema on origin: 2026-08-21
Horizons: [5, 10, 21] business days
Number of runs: 3


## Run Scenario Schema 3 Times

In [2]:
from aieng.forecasting.evaluation.task import ForecastingTask

results = []

# Task and context are built once — same origin, same information cutoff,
# for every run. No scoring against realized outcomes (this is a consistency
# test, not a backtest), so the origin can be as recent as today.
task = ForecastingTask(
    task_id="wti_forecast",
    target_series_id=WTI_SERIES_ID,
    horizons=HORIZONS,
    frequency="B",
    description="WTI price forecast",
)
context = data_service.context(as_of=ORIGIN_DATE)

for run_num in range(NUM_RUNS):
    print(f"\n{'='*72}")
    print(f"RUN {run_num + 1} / {NUM_RUNS}")
    print(f"{'='*72}")

    # Build a fresh predictor each run so nothing is cached/reused between runs.
    config = build_wti_news_scenario_schema_enhanced_config(model=LITE_MODEL)
    predictor = build_wti_scenario_schema_enhanced_predictor(config)

    predictions = predictor.predict(task, context)

    results.append({
        "run_num": run_num + 1,
        "predictions": predictions,
    })

    for pred in predictions:
        if isinstance(pred.payload, ContinuousForecast):
            print(f"  point=${pred.payload.point_forecast:.2f}")

print(f"\n✓ All {NUM_RUNS} runs complete")


RUN 1 / 3


/home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/google/adk/tools/function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


  point=$88.50
  point=$89.50
  point=$87.00

RUN 2 / 3
  point=$89.50
  point=$92.00
  point=$95.00

RUN 3 / 3
  point=$89.50
  point=$91.00
  point=$93.00

✓ All 3 runs complete


In [3]:
print("="*72)
print("FACTORS AND SCENARIOS COMPARISON ACROSS RUNS")
print("="*72)
print(f"Testing Scenario Schema on origin: {ORIGIN_DATE.date()}")

for run_data in results:
    run_num = run_data["run_num"]
    predictions = run_data["predictions"]

    pred = predictions[0] if predictions else None

    if not pred or not pred.metadata:
        continue

    metadata = pred.metadata
    factors = metadata.get("factors", [])
    scenarios = metadata.get("scenarios", [])
    rationale = metadata.get("rationale", "")

    print(f"\n{'─'*72}")
    print(f"RUN {run_num}")
    print(f"{'─'*72}")

    # Display factors
    print(f"\nFACTORS ({len(factors)}):")
    for factor in factors:
        name = factor.get('name')
        tier = factor.get('tier')
        impact = factor.get('impact_score', 'N/A')
        impact_str = f", impact={impact}" if tier == "transitory" else ""
        print(f"  • {name} (tier={tier}{impact_str})")

    # Display scenarios
    print(f"\nSCENARIOS ({len(scenarios)}):")
    for scenario in scenarios:
        name = scenario.get('name')
        prob = scenario.get('probability')
        low = scenario.get('price_low')
        high = scenario.get('price_high')
        tail = scenario.get('is_tail_case')
        tail_str = " [TAIL]" if tail else ""
        print(f"  • {name}{tail_str}: P={prob:.1%}, Price ${low:.2f}–${high:.2f}")

    # Display rationale
    print(f"\nRATIONALE:\n  {rationale}")

FACTORS AND SCENARIOS COMPARISON ACROSS RUNS
Testing Scenario Schema on origin: 2026-08-21

────────────────────────────────────────────────────────────────────────
RUN 1
────────────────────────────────────────────────────────────────────────

FACTORS (3):
  • Strait of Hormuz Geopolitical Risk (tier=core)
  • Global Supply Chain/Tanker Constraints (tier=core)
  • US Maximum Pressure Campaign (tier=transitory, impact=high)

SCENARIOS (3):
  • Escalation/Supply Squeeze: P=35.0%, Price $90.00–$110.00
  • Diplomatic De-escalation: P=45.0%, Price $75.00–$85.00
  • Black Swan: Total Strait Closure [TAIL]: P=20.0%, Price $115.00–$140.00

RATIONALE:
  The market is currently driven by a high-stakes geopolitical standoff. The forecast reflects a cautious outlook where the risk of further escalation (the 'Escalation' scenario) is balanced against the possibility of a diplomatic cooling (the 'De-escalation' scenario). The 'Black Swan' scenario accounts for the extreme tail risk of a total closu

## Compare Point Forecasts Across Runs

In [4]:
# Extract point forecasts for each horizon across all runs
forecast_comparison = {h: [] for h in HORIZONS}

for run_data in results:
    predictions = run_data["predictions"]

    for pred in predictions:
        if isinstance(pred.payload, ContinuousForecast):
            # Find which horizon this is
            as_of = pd.Timestamp(pred.as_of)
            forecast_date = pd.Timestamp(pred.forecast_date)
            offset = pd.tseries.offsets.BDay()

            for h in HORIZONS:
                target_date = as_of + offset * h
                if target_date.normalize() == forecast_date.normalize():
                    forecast_comparison[h].append(pred.payload.point_forecast)
                    break

# Display comparison table
comparison_rows = []
for h in HORIZONS:
    forecasts = forecast_comparison[h]
    if forecasts:
        row = {
            "Horizon": f"{h}d",
            "Run 1": f"${forecasts[0]:.2f}" if len(forecasts) > 0 else "—",
            "Run 2": f"${forecasts[1]:.2f}" if len(forecasts) > 1 else "—",
            "Run 3": f"${forecasts[2]:.2f}" if len(forecasts) > 2 else "—",
            "Mean": f"${np.mean(forecasts):.2f}",
            "Std Dev": f"${np.std(forecasts):.2f}",
            "Range": f"${np.max(forecasts) - np.min(forecasts):.2f}",
        }
        comparison_rows.append(row)

df_comparison = pd.DataFrame(comparison_rows)
print("\n" + "="*72)
print("POINT FORECAST COMPARISON ACROSS 3 RUNS")
print("="*72)
print(df_comparison.to_string(index=False))


POINT FORECAST COMPARISON ACROSS 3 RUNS
Horizon  Run 1  Run 2  Run 3   Mean Std Dev Range
     5d $88.50 $89.50 $89.50 $89.17   $0.47 $1.00
    10d $89.50 $92.00 $91.00 $90.83   $1.03 $2.50
    21d $87.00 $95.00 $93.00 $91.67   $3.40 $8.00


## Extract Full Distributions (Quantiles)

In [9]:
# Extract full distributions and build comparison table
all_distributions = {run_idx: {h: None for h in HORIZONS} for run_idx in range(NUM_RUNS)}

for run_idx, run_data in enumerate(results):
    predictions = run_data["predictions"]

    for pred in predictions:
        if isinstance(pred.payload, ContinuousForecast):
            cf = pred.payload
            as_of = pd.Timestamp(pred.as_of)
            forecast_date = pd.Timestamp(pred.forecast_date)
            offset = pd.tseries.offsets.BDay()

            for h in HORIZONS:
                target_date = as_of + offset * h
                if target_date.normalize() == forecast_date.normalize():
                    if all_distributions[run_idx][h] is None:
                        all_distributions[run_idx][h] = cf
                    break

# Build table
dist_rows = []
for h in HORIZONS:
    row = {"Horizon": f"{h}d"}
    for run_idx in range(NUM_RUNS):
        cf = all_distributions[run_idx][h]
        if cf is not None:
            point = f"${cf.point_forecast:.2f}"

            # Try to get CI
            # 80% CI from the quantiles dict (keyed by quantile level, e.g. 0.1, 0.9)
            ci = ""
            if cf.quantiles:
                q10 = cf.quantiles.get(0.1)
                q90 = cf.quantiles.get(0.9)
                if q10 is not None and q90 is not None:
                    ci = f" [{q10:.2f}, {q90:.2f}]"

            row[f"Run {run_idx+1}"] = point + ci
    dist_rows.append(row)

df_distributions = pd.DataFrame(dist_rows)
print("\n" + "="*72)
print("FULL DISTRIBUTIONS BY HORIZON (point + 80% CI)")
print("="*72)
print(df_distributions.to_string(index=False))


FULL DISTRIBUTIONS BY HORIZON (point + 80% CI)
Horizon                 Run 1                  Run 2                  Run 3
     5d $88.50 [85.20, 94.50]  $89.50 [85.50, 97.00]  $89.50 [85.50, 97.00]
    10d $89.50 [84.00, 97.00] $92.00 [84.00, 104.00] $91.00 [84.00, 102.00]
    21d $87.00 [80.50, 97.00] $95.00 [81.00, 113.00] $93.00 [81.00, 110.00]


## Extract Rationales (Agent Reasoning)

In [6]:
# Extract rationales from prediction metadata - deduplicated
for run_idx, run_data in enumerate(results):
    print(f"\n{'='*72}")
    print(f"RUN {run_idx + 1} — AGENT RATIONALE")
    print(f"{'='*72}\n")

    predictions = run_data["predictions"]
    seen_rationales = set()

    for pred in predictions:
        # Check for rationale in metadata
        if hasattr(pred, 'metadata') and pred.metadata:
            rationale = pred.metadata.get('rationale', None)
            if rationale and rationale not in seen_rationales:
                print(rationale)
                print()
                seen_rationales.add(rationale)

        # Also check payload for any reasoning field
        if hasattr(pred.payload, 'reasoning'):
            reasoning = pred.payload.reasoning
            if reasoning and reasoning not in seen_rationales:
                print(reasoning)
                print()
                seen_rationales.add(reasoning)


RUN 1 — AGENT RATIONALE

The market is currently driven by a high-stakes geopolitical standoff. The forecast reflects a cautious outlook where the risk of further escalation (the 'Escalation' scenario) is balanced against the possibility of a diplomatic cooling (the 'De-escalation' scenario). The 'Black Swan' scenario accounts for the extreme tail risk of a total closure of the Strait of Hormuz, which would cause a massive, immediate supply shock.


RUN 2 — AGENT RATIONALE

The market is currently in a high-tension state driven by the threat of disruption in the Strait of Hormuz. The forecast reflects a significant risk premium that is highly sensitive to news flow regarding U.S.-Iran relations. The scenarios capture the binary risk of a major supply disruption versus a potential, albeit less likely, diplomatic de-escalation.


RUN 3 — AGENT RATIONALE

The market is currently in a high-stakes standoff. The core drivers are the structural tightness of global oil inventories and the per

## Consistency Assessment

In [7]:
print("\n" + "="*72)
print("CONSISTENCY METRICS")
print("="*72)

for h in HORIZONS:
    forecasts = forecast_comparison[h]
    if len(forecasts) == NUM_RUNS:
        mean = np.mean(forecasts)
        std = np.std(forecasts)
        cv = (std / mean) * 100
        print(f"\nh={h}d:")
        print(f"  Mean:                 ${mean:.2f}")
        print(f"  Std Dev:              ${std:.2f}")
        print(f"  Coefficient of Var:   {cv:.1f}%")
        print(f"  Range (max - min):    ${np.max(forecasts) - np.min(forecasts):.2f}")
        
        if cv < 1.0:
            consistency = "✓ Very consistent (CV < 1%)" 
        elif cv < 2.0:
            consistency = "✓ Consistent (CV 1-2%)"
        elif cv < 5.0:
            consistency = "⚠ Moderate variance (CV 2-5%)"
        else:
            consistency = "✗ High variance (CV > 5%)"
        print(f"  Assessment:           {consistency}")


CONSISTENCY METRICS

h=5d:
  Mean:                 $89.17
  Std Dev:              $0.47
  Coefficient of Var:   0.5%
  Range (max - min):    $1.00
  Assessment:           ✓ Very consistent (CV < 1%)

h=10d:
  Mean:                 $90.83
  Std Dev:              $1.03
  Coefficient of Var:   1.1%
  Range (max - min):    $2.50
  Assessment:           ✓ Consistent (CV 1-2%)

h=21d:
  Mean:                 $91.67
  Std Dev:              $3.40
  Coefficient of Var:   3.7%
  Range (max - min):    $8.00
  Assessment:           ⚠ Moderate variance (CV 2-5%)


In [8]:
import json

# Full metadata dump per run — factors/scenarios are already surfaced in cell-5
# above; this is the raw structured payload for anyone who wants everything.
for run_idx, run_data in enumerate(results):
    print(f"\n{'='*72}")
    print(f"RUN {run_idx + 1} — FULL PREDICTION METADATA")
    print(f"{'='*72}\n")

    predictions = run_data["predictions"]
    seen = set()

    for pred in predictions:
        if hasattr(pred, 'metadata') and pred.metadata:
            metadata_str = json.dumps(pred.metadata, indent=2, default=str)
            if metadata_str not in seen:
                print(metadata_str)
                print()
                seen.add(metadata_str)


RUN 1 — FULL PREDICTION METADATA

{
  "factors": [
    {
      "name": "Strait of Hormuz Geopolitical Risk",
      "category": "geopolitical",
      "tier": "core",
      "impact_score": null
    },
    {
      "name": "Global Supply Chain/Tanker Constraints",
      "category": "macro",
      "tier": "core",
      "impact_score": null
    },
    {
      "name": "US Maximum Pressure Campaign",
      "category": "geopolitical",
      "tier": "transitory",
      "impact_score": "high"
    }
  ],
  "scenarios": [
    {
      "name": "Escalation/Supply Squeeze",
      "probability": 0.35,
      "price_low": 90.0,
      "price_high": 110.0,
      "is_tail_case": false,
      "stances": {
        "Strait of Hormuz Geopolitical Risk": "bullish",
        "Global Supply Chain/Tanker Constraints": "bullish",
        "US Maximum Pressure Campaign": "bullish"
      }
    },
    {
      "name": "Diplomatic De-escalation",
      "probability": 0.45,
      "price_low": 75.0,
      "price_high": 85.0,